In [2]:
import os

In [3]:
%pwd

'c:\\Users\\Divya Naidu\\Desktop\\Code\\projects\\kidneyDisease\\Kidney-Disease-Classification\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'c:\\Users\\Divya Naidu\\Desktop\\Code\\projects\\kidneyDisease\\Kidney-Disease-Classification'

In [6]:
#writing entity here
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list


In [7]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories
import tensorflow as tf

In [13]:
class ConfigurationManager:

    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_training_config(self) -> TrainingConfig:

        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params

        training_data = os.path.join(
            self.config.data_ingestion.unzip_dir,
            "CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone"
        )

        create_directories([
            Path(training.root_dir)
        ])

        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(
                prepare_base_model.updated_base_model_path
            ),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size=params.BATCH_SIZE,
            params_is_augmentation=params.AUGMENTATION,
            params_image_size=params.IMAGE_SIZE,
        )

        return training_config

In [14]:
config = ConfigurationManager()

[2026-08-12 21:06:54,839: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-08-12 21:06:54,844: INFO: common: yaml file: params.yaml loaded successfully]
[2026-08-12 21:06:54,845: INFO: common: created directory at: artifacts]


In [9]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [20]:
class Training:

    def __init__(self, config: TrainingConfig):
        self.config = config

    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )

        self.model.compile(
            optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
            loss=tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )

    def train_valid_generator(self):

        data_generator_kwargs = dict(
            rescale=1.0 / 255,
            validation_split=0.2
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **data_generator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:

            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=20,
                horizontal_flip=True,
                vertical_flip=True,
                width_shift_range=0.1,
                height_shift_range=0.1,
                zoom_range=0.2,
                **data_generator_kwargs
            )

        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    def train(self):

        self.steps_per_epoch = (
            self.train_generator.samples //
            self.train_generator.batch_size
        )

        self.validation_steps = (
            self.valid_generator.samples //
            self.valid_generator.batch_size
        )

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )

In [ ]:
training_config = config.get_training_config()

print("Model path:")
print(training_config.updated_base_model_path)

print("\nTraining data path:")
print(training_config.training_data)

[2026-08-12 21:07:05,650: INFO: common: created directory at: artifacts\training]
Model path:
artifacts\prepare_base_model\base_model_updated.h5

Training data path:
artifacts\data_ingestion\CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone


In [16]:
training = Training(config=training_config)

training.get_base_model()
training.train_valid_generator()

[2026-08-12 21:07:26,696: WARNING: config: TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.]
[2026-08-12 21:07:26,701: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Found 1471 images belonging to 2 classes.
Found 5889 images belonging to 2 classes.


In [22]:
try:

    training_config = config.get_training_config()

    print("Model path:")
    print(training_config.updated_base_model_path)

    print("\nTraining data path:")
    print(training_config.training_data)

    training = Training(
        config=training_config
    )

    training.get_base_model()

    training.train_valid_generator()

    training.train()

except Exception as e:
    raise e

[2026-08-12 22:11:15,022: INFO: common: created directory at: artifacts\training]
Model path:
artifacts\prepare_base_model\base_model_updated.h5

Training data path:
artifacts\data_ingestion\CT-KIDNEY-DATASET-Normal-Cyst-Tumor-Stone
[2026-08-12 22:11:15,382: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Found 1471 images belonging to 2 classes.
Found 5889 images belonging to 2 classes.
736/736 ━━━━━━━━━━━━━━━━━━━━ 1785s 2s/step - accuracy: 0.6497 - loss: 0.6408 - val_accuracy: 0.6933 - val_loss: 0.6172
[2026-08-12 22:41:01,238: WARNING: saving_api: You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. ]
